<a href="https://colab.research.google.com/github/ronan-cunha/machine-learning-course/blob/main/Notebooks/aula_8_avaliacao_modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold

In [2]:
# Carrega o dataset Fair (sobre casos extraconjugais)
# https://www.statsmodels.org/stable/datasets/generated/fair.html
affair = sm.datasets.fair.load_pandas()
df_real = affair.data
display(df_real.head())

,rate_marriage,age,yrs_married,children,religious,educ,occupation,occupation_husb,affairs
0,3.0,32.0,9.0,3.0,3.0,17.0,2.0,5.0,0.111111
1,3.0,27.0,13.0,3.0,1.0,14.0,3.0,4.0,3.230769
2,4.0,22.0,2.5,0.0,1.0,16.0,3.0,5.0,1.400000
3,4.0,37.0,16.5,4.0,3.0,16.0,5.0,5.0,0.727273
4,5.0,27.0,9.0,1.0,1.0,14.0,3.0,4.0,4.666666


In [19]:
# A variável dependente binária 'had_affair'(1 é sim e 0 é não)
df_real['had_affair'] = (affair.endog > 0).astype(int)
df_real = df_real[['had_affair','rate_marriage','age','educ']]
display(df_real.head())

,had_affair,rate_marriage,age,educ
0,1,3.0,32.0,17.0
1,1,3.0,27.0,14.0
2,1,4.0,22.0,16.0
3,1,4.0,37.0,16.0
4,1,5.0,27.0,14.0


## Amostra de teste e treino

In [21]:
num_splits = 4
kf = KFold(n_splits=num_splits, shuffle=True, random_state=42)
kf

KFold(n_splits=4, random_state=42, shuffle=True)

In [37]:
kf_iterator = iter(kf.split(df_real))
train_index_0, test_index_0 = next(kf_iterator)
print(f"Tamanho do conjunto de treino: {len(train_index_0)}")
print(f"Tamanho do conjunto de teste: {len(test_index_0)}")
display(df_real.iloc[train_index_0].describe())
display(df_real.iloc[test_index_0].describe())

Tamanho do conjunto de treino: 4774
Tamanho do conjunto de teste: 1592


,had_affair,rate_marriage,age,educ
count,4774.000000,4774.000000,4774.000000,4774.000000
mean,0.325513,4.106829,29.122329,14.198785
std,0.468615,0.963999,6.882367,2.165785
min,0.000000,1.000000,17.500000,9.000000
25%,0.000000,4.000000,22.000000,12.000000
50%,0.000000,4.000000,27.000000,14.000000
75%,1.000000,5.000000,32.000000,16.000000
max,1.000000,5.000000,42.000000,20.000000


,had_affair,rate_marriage,age,educ
count,1592.000000,1592.000000,1592.000000,1592.000000
mean,0.313442,4.118090,28.964510,14.243090
std,0.464038,0.953936,6.744169,2.214591
min,0.000000,1.000000,17.500000,9.000000
25%,0.000000,4.000000,22.000000,12.000000
50%,0.000000,4.000000,27.000000,14.000000
75%,1.000000,5.000000,32.000000,16.000000
max,1.000000,5.000000,42.000000,20.000000


In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error


class RegressionModel:
    def __init__(self, model_type):
        self.model_type = model_type
        self.model = None

    def fit(self, formula, data):
        if self.model_type == 'LPM':
            self.model = smf.ols(formula, data=data).fit()
        elif self.model_type == 'Logit':
            self.model = smf.logit(formula, data=data).fit()
        elif self.model_type == 'Probit':
            self.model = smf.probit(formula, data=data).fit()
        else:
            raise ValueError("Tipo de modelo inválido. Escolha 'LPM', 'Logit' ou 'Probit'.")

    def predict(self, data):
        if self.model:
            return self.model.predict(data)
        else:
            raise ValueError("O modelo não foi ajustado ainda. Chame .fit() primeiro.")


print("Classe RegressionModel e função de métricas definidas.")

In [ ]:
# Definindo a fórmula do modelo
formula = 'had_affair ~ rate_marriage + age + educ'

# Listas para armazenar os resultados das métricas
results = {
    'LPM': {'MSE': [], 'MAE': [], 'MAPE': []},
    'Logit': {'MSE': [], 'MAE': [], 'MAPE': []},
    'Probit': {'MSE': [], 'MAE': [], 'MAPE': []},
}

num_splits = 4
kf = KFold(n_splits=num_splits, shuffle=True, random_state=42)

for i, (train_index, test_index) in enumerate(kf.split(df_real)):
    print(f"\n--- Amostra de Treino/Teste {i+1}/{num_splits} ---")

    # Dividir os dados em treino e teste (usando KFold para splits não sobrepostos)
    df_train = df_real.iloc[train_index]
    df_test = df_real.iloc[test_index]

    # Instanciar e ajustar os modelos
    lpm_classifier = RegressionModel('LPM')
    lpm_classifier.fit(formula, df_train)

    logit_classifier = RegressionModel('Logit')
    logit_classifier.fit(formula, df_train)

    probit_classifier = RegressionModel('Probit')
    probit_classifier.fit(formula, df_train)

    # Fazer previsões no conjunto de teste
    y_true = df_test['had_affair']

    lpm_preds = lpm_classifier.predict(df_test)
    logit_preds = logit_classifier.predict(df_test)
    probit_preds = probit_classifier.predict(df_test)

    # Avaliar o LPM
    results['LPM']['MSE'].append(mean_squared_error(y_true, lpm_preds))
    results['LPM']['MAE'].append(mean_absolute_error(y_true, lpm_preds))
    results['LPM']['MAPE'].append(mean_absolute_percentage_error(y_true, lpm_preds))

    # Avaliar o Logit
    results['Logit']['MSE'].append(mean_squared_error(y_true, logit_preds))
    results['Logit']['MAE'].append(mean_absolute_error(y_true, logit_preds))
    results['Logit']['MAPE'].append(mean_absolute_percentage_error(y_true, logit_preds))

    # Avaliar o Probit
    results['Probit']['MSE'].append(mean_squared_error(y_true, probit_preds))
    results['Probit']['MAE'].append(mean_absolute_error(y_true, probit_preds))
    results['Probit']['MAPE'].append(mean_absolute_percentage_error(y_true, probit_preds))


print("\n--- Avaliação Completa ---")

In [ ]:
# Exibir os resultados médios das métricas
summary_results = {}
for model_name, metrics in results.items():
    summary_results[model_name] = {
        'Mean MSE': np.mean(metrics['MSE']),
        'Mean MAE': np.mean(metrics['MAE']),
        'Mean MAPE': np.mean(metrics['MAPE'])
    }

summary_df = pd.DataFrame(summary_results).T
print("\nMétricas de Avaliação (Média sobre as amostras):")
display(summary_df)
